# Load category distribution information
the statistic information about arXiv is stored in `category_count./csv`

In [ ]:
import csv


def load_csv(file):
    data = []
    with open(file, newline='') as csvfile:
        cr = csv.DictReader(csvfile)
        for row in cr:
            data.append(row)

    return data


file = 'category_count.csv'
data = load_csv(file)

# Download source files of papers by 

In [ ]:
import arxiv
import os
from typing import List, Dict


def arxiv_download(data: List[Dict]):
    papers = set()

    for row in data[::-1]:
        category, count = row["categories"], int(row["count"])
        source_directory = os.path.expanduser("~/vrdu_data" + "/" + category)
        os.makedirs(source_directory, exist_ok=True)

        if len(os.listdir(source_directory)) > count:
            print(f"Complete process for {category}, {count} papers")
            continue

        search = arxiv.Search(
            query=category,
            max_results=count + 20,
            sort_by=arxiv.SortCriterion.SubmittedDate,
        )

        for result in search.results():
            if len(os.listdir(source_directory)) > count:
                break

            if result.entry_id in papers:
                continue
            papers.add(result.entry_id)
            result.download_source(dirpath=source_directory)

        print(f"Complete process for {category}, {count} papers")

# arxiv_download(data)

# Process
After downloading the source files of PDFs, we first unzip them, then we run the scripts to generate the annotation result, finaly, we export the result

In [ ]:
import os
import tarfile


def extract_all_tar_gz(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".tar.gz"):
                file_path = os.path.join(root, file)
                extract_path = os.path.splitext(file_path)[0]
                extract_tar_gz(file_path, extract_path)


def extract_tar_gz(file_path, extract_path):
    with tarfile.open(file_path, 'r:gz') as tar:
        tar.extractall(extract_path)


source_directory = os.path.expanduser("~/vrdu_data")
extract_all_tar_gz(source_directory)

In [ ]:
# find all tex files
import os


def find_tex_files(directory):
    tex_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".tex"):
                file_path = os.path.join(root, file)
                tex_files.append(file_path)
    return tex_files


# Usage example
source_directory = os.path.expanduser("~/vrdu_data")

tex_files = find_tex_files(source_directory)

# Preprocess

In [ ]:
print("before processing, there are {} tex files".format(len(tex_files)))
result = []
for tex_file in tex_files:
    print(f"processing {tex_file}")
    try:
        tex_text = open(tex_file).read()
        if tex_text.find(r"\begin{document}") == -1:
            continue
        result.append(tex_file)
    except:
        pass

print("after processing, there are {} tex files".format(len(result)))
tex_files = result

In [7]:
# run vrdu_data process for all files:
import tqdm
import subprocess
import random

script_path = os.path.expanduser("./pipeline.sh")
random.shuffle(tex_files)
output_file = "output.txt"
time_out = 180

for tex_file in tqdm.tqdm(tex_files):
    print(f"processing {tex_file}")
    try:
        # Execute the shell script for each object
        subprocess.run(['bash', script_path, tex_file], check=True,
                       stdout=open(output_file, 'w'),
                       timeout=time_out)
        print(f"Script executed successfully for {tex_file}")
    except subprocess.CalledProcessError as e:
        # Handle any exceptions that occur during script execution
        print(f"Script failed for {tex_file}: {e}")
    except subprocess.TimeoutExpired as e:
        # Handle any exceptions that occur during script execution
        print(f"Script timed out for {tex_file}: {e}")
    else:
        # Script executed successfully
        print(f"Script executed successfully for {tex_file}")

  0%|          | 0/356 [00:00<?, ?it/s]

processing /home/PJLAB/maosong/vrdu_data/cond-mat.supr-con/2309.11874v1.Cuprate_universal_electronic_spin_response_and_the_pseudogap_from_NMR.tar/BandurEtAl.tex


Traceback (most recent call last):
  File "run_rendering.py", line 103, in <module>
    main()
  File "run_rendering.py", line 91, in main
    data = utils.data_from_tex_file(origin_tex_file, debug_mode)
  File "/home/PJLAB/maosong/vrdu_data_process/rendering/utils.py", line 76, in data_from_tex_file
    tex_tree = TexSoup(tex_text).expr.all
  File "/home/PJLAB/maosong/vrdu_data_process/TexSoup/TexSoup/__init__.py", line 87, in TexSoup
    parsed, src = read(tex_code, skip_envs=skip_envs, tolerance=tolerance)
  File "/home/PJLAB/maosong/vrdu_data_process/TexSoup/TexSoup/tex.py", line 22, in read
    return TexEnv('[tex]', begin='', end='', contents=buf), tex
  File "/home/PJLAB/maosong/vrdu_data_process/TexSoup/TexSoup/data.py", line 897, in __init__
    super().__init__(name, contents, args, preserve_whitespace, position)
  File "/home/PJLAB/maosong/vrdu_data_process/TexSoup/TexSoup/data.py", line 646, in __init__
    self._contents = list(contents) or []
  File "/home/PJLAB/maosong/v

Script failed for /home/PJLAB/maosong/vrdu_data/cond-mat.supr-con/2309.11874v1.Cuprate_universal_electronic_spin_response_and_the_pseudogap_from_NMR.tar/BandurEtAl.tex: Command '['bash', './pipeline.sh', '/home/PJLAB/maosong/vrdu_data/cond-mat.supr-con/2309.11874v1.Cuprate_universal_electronic_spin_response_and_the_pseudogap_from_NMR.tar/BandurEtAl.tex']' returned non-zero exit status 1.
processing /home/PJLAB/maosong/vrdu_data/cs.LG/2309.11601v1.Latent_Diffusion_Models_for_Structural_Component_Design.tar/Old_TeX/SMI2023.tex


  1%|          | 2/356 [03:00<10:26:42, 106.22s/it]

...

Script failed for /home/PJLAB/maosong/vrdu_data/cond-mat.other/2309.12144v1.Towards_a_minimal_description_of_dynamical_correlation_in_metals.tar/U_om.tex: Command '['bash', './pipeline.sh', '/home/PJLAB/maosong/vrdu_data/cond-mat.other/2309.12144v1.Towards_a_minimal_description_of_dynamical_correlation_in_metals.tar/U_om.tex']' returned non-zero exit status 1.
processing /home/PJLAB/maosong/vrdu_data/q-fin.PR/2309.08175v1.A_Markovian_empirical_model_for_the_VIX_index_and_the_pricing_of_the_corresponding_derivatives.tar/submitted_to_journal_of_derivatives.tex


Traceback (most recent call last):
  File "generate_bb.py", line 293, in <module>
    main()
  File "generate_bb.py", line 267, in main
    transformed_page_elements = geometry.transform(page_elements, page_image)
  File "/home/PJLAB/maosong/vrdu_data_process/layout/geometry.py", line 227, in transform
    raise Exception("image size and page size are not scaled")
Exception: image size and page size are not scaled
  4%|▍         | 15/356 [19:07<5:11:13, 54.76s/it]

Script failed for /home/PJLAB/maosong/vrdu_data/q-fin.PR/2309.08175v1.A_Markovian_empirical_model_for_the_VIX_index_and_the_pricing_of_the_corresponding_derivatives.tar/submitted_to_journal_of_derivatives.tex: Command '['bash', './pipeline.sh', '/home/PJLAB/maosong/vrdu_data/q-fin.PR/2309.08175v1.A_Markovian_empirical_model_for_the_VIX_index_and_the_pricing_of_the_corresponding_derivatives.tar/submitted_to_journal_of_derivatives.tex']' returned non-zero exit status 1.
processing /home/PJLAB/maosong/vrdu_data/cs.LG/2309.11702v1.Incentivized_Communication_for_Federated_Bandits.tar/main.tex


  4%|▍         | 16/356 [19:25<4:07:17, 43.64s/it]

Script executed successfully for /home/PJLAB/maosong/vrdu_data/cs.LG/2309.11702v1.Incentivized_Communication_for_Federated_Bandits.tar/main.tex
Script executed successfully for /home/PJLAB/maosong/vrdu_data/cs.LG/2309.11702v1.Incentivized_Communication_for_Federated_Bandits.tar/main.tex
processing /home/PJLAB/maosong/vrdu_data/cs.LG/2309.12032v1.Human_in_the_Loop_Causal_Discovery_under_Latent_Confounding_using_Ancestral_GFlowNets.tar/tikzpeople.tex


In [ ]:
import os
import shutil
import zipfile


def extract_result(source_directory, destination_directory):
    # Create the destination directory if it doesn't exist
    os.makedirs(destination_directory, exist_ok=True)

    # Iterate over the nested directories
    for root, dirs, files in os.walk(source_directory):
        if "output" in dirs and "result" in os.listdir(os.path.join(root, "output")):
            # Construct the path to the "result" subdirectory
            result_directory = os.path.join(root, "output", "result")

            # Create a new directory in the destination directory
            new_directory = os.path.join(
                destination_directory, os.path.basename(root))
            os.makedirs(new_directory, exist_ok=True)

            # Copy the "result" subdirectory to the new directory
            shutil.copytree(
                result_directory, os.path.join(
                    new_directory, os.path.basename(root))
            )

            # Zip the new directory
            zip_file_path = os.path.join(
                destination_directory, f"{os.path.basename(root)}.zip"
            )
            with zipfile.ZipFile(zip_file_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
                for folder_name, _, file_names in os.walk(new_directory):
                    for file_name in file_names:
                        file_path = os.path.join(folder_name, file_name)
                        zip_file.write(
                            file_path, os.path.relpath(
                                file_path, new_directory)
                        )

            # Remove the copied "result" subdirectory
            shutil.rmtree(new_directory)


destination_directory = os.path.expanduser("~/vrdu_data/annotations")
os.mkdir(destination_directory)
extract_result(source_directory, destination_directory)